In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_csv("Dataset_B.csv")

# FORWARD: geometry is input, modal features are output
input_cols  = ['r1','r2','r3','D1','D2','D3']
target_cols = [c for c in df.columns if c not in input_cols]

X = df[input_cols]
y = df[target_cols]

In [3]:
# Impute X
imputer_X = SimpleImputer(strategy="median")
X_imputed = imputer_X.fit_transform(X)

# For Y replace NaN with 0
y_filled = np.nan_to_num(y.values, nan=0.0)

# Remove low variance columns
selector = VarianceThreshold(threshold=1e-10)
y_imputed = selector.fit_transform(y_filled)

print("Output features after variance filter:", y_imputed.shape[1])

Output features after variance filter: 8


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y_imputed, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (3918, 6)
Test size: (980, 6)


In [5]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    n_jobs=-1,
    random_state=42
)

model = MultiOutputRegressor(rf)
print("Training Forward Random Forest...")
model.fit(X_train, y_train)
print("Training Complete!")

Training Forward Random Forest...
Training Complete!


In [6]:
y_pred = model.predict(X_test)

print("===== Forward Random Forest Evaluation =====")
for i in range(6):
    r2   = r2_score(y_test[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))
    mae  = mean_absolute_error(y_test[:, i], y_pred[:, i])
    print(f"Feature {i+1}: RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}")

overall_r2 = r2_score(y_test.flatten(), y_pred.flatten())
print(f"\nOverall R²: {overall_r2*100:.2f}%")

===== Forward Random Forest Evaluation =====
Feature 1: RMSE=0.0007 | MAE=0.0004 | R²=0.9917
Feature 2: RMSE=0.0008 | MAE=0.0005 | R²=0.9885
Feature 3: RMSE=0.0008 | MAE=0.0005 | R²=0.9886
Feature 4: RMSE=0.0009 | MAE=0.0006 | R²=0.9874
Feature 5: RMSE=0.0009 | MAE=0.0006 | R²=0.9874
Feature 6: RMSE=0.0009 | MAE=0.0006 | R²=0.9866

Overall R²: 100.00%


In [7]:
np.save("true_forward_RF.npy", y_test)
np.save("pred_forward_RF.npy", y_pred)
print("Saved!")

Saved!
